# Room 1: Templates, Registration, and OCR

This notebook walks through the classic template-based document extraction pipeline:

1. **Render** a synthetic employment application form
2. **Corrupt** it with realistic printer/scanner noise
3. **Register** the corrupted scan back to the template using ORB + RANSAC
4. **Crop** field regions and run OCR
5. **Evaluate** extraction accuracy

You'll see why this pipeline works on clean scans and how it breaks under real-world conditions.

> Companion to [Lessons from a Decade of Document AI Work](https://zzsi.github.io/blog/posts/20260331-practical-doc-ai/) — [Room 1 interactive exhibit](https://zzsi.github.io/blog/posts/20260331-practical-doc-ai/museum/room1-templates.html)

## 0. Setup

In [ ]:
!apt-get install -y tesseract-ocr -qq
!pip install -q python-doctr[torch] pytesseract

In [ ]:
from __future__ import annotations

import io
import math
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw, ImageFilter, ImageFont

SCALE = 3

def S(value: int | float) -> int:
    return int(round(value * SCALE))

TEMPLATE_SIZE = (S(700), S(1000))
PAGE_BOX = {"x": S(36), "y": S(22), "w": S(628), "h": S(972)}


@dataclass
class Field:
    id: str
    label: str
    value: str
    box: tuple[int, int, int, int]


@dataclass
class CorruptionProfile:
    id: str
    description: str
    print_artifacts: dict[str, Any]
    paper_artifacts: dict[str, Any]
    geometry: dict[str, Any]
    capture: dict[str, Any]


FIELDS = [
    Field("name", "Applicant name", "Alexandra Hayes", (S(110), S(224), S(300), S(30))),
    Field("position", "Position desired", "Senior Operations Analyst", (S(110), S(284), S(250), S(30))),
    Field("address", "Street address", "2714 Red Cedar Lane Larkhaven, TX 78705", (S(110), S(344), S(225), S(56))),
    Field("records_days", "Records due (days)", "15", (S(300), S(889), S(26), S(16))),
]

## 1. Rendering a Synthetic Form

Template-based extraction starts with a known form layout. Every field has a fixed pixel coordinate on the page. We render a synthetic employment application to use as our template.

In [ ]:
def load_font(name: str, size: int) -> ImageFont.FreeTypeFont:
    """Try platform-specific paths, fall back to default."""
    candidates = [
        # macOS
        Path("/System/Library/Fonts/Supplemental") / name,
        Path("/Library/Fonts") / name,
        Path("/System/Library/Fonts") / name,
        # Linux / Colab
        Path("/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()


FONT_TITLE = load_font("Georgia.ttf", S(21))
FONT_SUBTITLE = load_font("Georgia.ttf", S(11))
FONT_LABEL = load_font("Georgia.ttf", S(11))
FONT_VALUE = load_font("Georgia.ttf", S(13))
FONT_VALUE_SMALL = load_font("Georgia.ttf", S(11))
FONT_BODY = load_font("Georgia.ttf", S(10))
FONT_SECTION = load_font("Georgia.ttf", S(10))
FONT_TINY = load_font("Georgia.ttf", S(9))
FONT_MICRO = load_font("Georgia.ttf", S(8))

In [ ]:
def render_template(blank: bool = True) -> Image.Image:
    image = Image.new("RGB", TEMPLATE_SIZE, "#eee3ca")
    draw = ImageDraw.Draw(image)

    draw.rectangle((0, 0, TEMPLATE_SIZE[0] - 1, TEMPLATE_SIZE[1] - 1), fill="#eee3ca")
    px, py, pw, ph = PAGE_BOX["x"], PAGE_BOX["y"], PAGE_BOX["w"], PAGE_BOX["h"]
    draw.rectangle((px, py, px + pw, py + ph), fill="#f9f6eb", outline="#9d9278", width=2)

    draw.ellipse((S(94), S(72), S(146), S(124)), fill="#d7ccb4", outline="#8e836a", width=2)
    draw.text((S(164), S(74)), "Employment Application", font=FONT_TITLE, fill="#332b1d")
    draw.text((S(164), S(101)), "Northwind Logistics", font=FONT_SUBTITLE, fill="#5a533f")
    draw.text((S(164), S(121)), "Applicant intake form for hourly and salaried roles", font=FONT_BODY, fill="#5a533f")
    draw.text((S(164), S(136)), "Please print clearly. Attach supplemental work history if more space is needed.", font=FONT_TINY, fill="#6a624d")

    draw.rectangle((S(104), S(164), S(596), S(194)), fill="#e5ddca", outline="#9d9278", width=1)
    draw.text((S(116), S(171)), "Section 1. Applicant, Contact, and Role", font=FONT_SECTION, fill="#413828")

    draw.text((S(110), S(202)), "Applicant name", font=FONT_LABEL, fill="#574f40")
    draw.text((S(440), S(202)), "Preferred name / alias", font=FONT_LABEL, fill="#574f40")
    draw.rectangle((S(110), S(224), S(410), S(254)), outline="#756a52", width=1)
    draw.rectangle((S(440), S(224), S(586), S(254)), outline="#756a52", width=1)

    draw.text((S(110), S(262)), "Position desired", font=FONT_LABEL, fill="#574f40")
    draw.text((S(390), S(262)), "Department requested", font=FONT_LABEL, fill="#574f40")
    draw.rectangle((S(110), S(284), S(360), S(314)), outline="#756a52", width=1)
    draw.rectangle((S(390), S(284), S(586), S(314)), outline="#756a52", width=1)

    draw.text((S(110), S(322)), "Street address", font=FONT_LABEL, fill="#574f40")
    draw.text((S(360), S(322)), "City / State / ZIP", font=FONT_LABEL, fill="#574f40")
    draw.rectangle((S(110), S(344), S(335), S(400)), outline="#756a52", width=1)
    draw.rectangle((S(360), S(344), S(586), S(376)), outline="#756a52", width=1)

    contact_labels = [("Mobile phone", S(110)), ("Alternate phone", S(255)), ("Emergency phone", S(430))]
    for text, x in contact_labels:
        draw.text((x, S(414)), text, font=FONT_BODY, fill="#6b6557")
    draw.line((S(110), S(448), S(230), S(448)), fill="#b0a58f", width=1)
    draw.line((S(255), S(448), S(405), S(448)), fill="#b0a58f", width=1)
    draw.line((S(430), S(448), S(585), S(448)), fill="#b0a58f", width=1)

    draw.text((S(110), S(458)), "Email address", font=FONT_BODY, fill="#6b6557")
    draw.text((S(345), S(458)), "Current supervisor", font=FONT_BODY, fill="#6b6557")
    draw.line((S(110), S(486), S(320), S(486)), fill="#b0a58f", width=1)
    draw.line((S(345), S(486), S(586), S(486)), fill="#b0a58f", width=1)

    if not blank:
        draw.text((S(118), S(231)), "Alexandra Hayes", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(448), S(231)), "Alex Hayes", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(118), S(291)), "Senior Operations Analyst", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(398), S(291)), "Network Operations", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(118), S(359)), "2714 Red Cedar Lane", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(118), S(377)), "Larkhaven, TX 78705", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(368), S(352)), "Larkhaven, TX 78705", font=FONT_VALUE_SMALL, fill="#2f2b23")
        draw.text((S(110), S(428)), "(512) 555-0147", font=FONT_BODY, fill="#2f2b23")
        draw.text((S(255), S(428)), "(737) 555-0199", font=FONT_BODY, fill="#2f2b23")
        draw.text((S(430), S(428)), "(512) 555-0102", font=FONT_BODY, fill="#2f2b23")
        draw.text((S(110), S(470)), "alex.hayes@example.com", font=FONT_BODY, fill="#2f2b23")
        draw.text((S(345), S(470)), "M. Patel / (512) 555-0130", font=FONT_BODY, fill="#2f2b23")

    draw.rectangle((S(104), S(500), S(596), S(530)), fill="#e5ddca", outline="#9d9278", width=1)
    draw.text((S(116), S(507)), "Section 2. Recent Work History", font=FONT_SECTION, fill="#413828")

    history_rows = [
        {"y": 546, "employer": "Northwind Fulfillment", "address": "9101 Logistics Way, Brighthaven, TX",
         "dates": "03/2022 - 02/2026", "title": "Operations Coordinator", "supervisor": "J. Rivera",
         "reason": "Relocation of facility"},
        {"y": 660, "employer": "Capstone Field Services", "address": "4800 Mesa Park Dr, Stonecross, TX",
         "dates": "07/2019 - 02/2022", "title": "Customer Success Lead", "supervisor": "L. Chen",
         "reason": "Pursued internal operations role"},
    ]
    for row in history_rows:
        y = S(row["y"])
        draw.text((S(110), y), "Employer name", font=FONT_BODY, fill="#6b6557")
        draw.text((S(280), y), "Employer address", font=FONT_BODY, fill="#6b6557")
        draw.text((S(500), y), "Dates employed", font=FONT_BODY, fill="#6b6557")
        draw.text((S(110), y + S(40)), "Job title", font=FONT_BODY, fill="#6b6557")
        draw.text((S(360), y + S(40)), "Supervisor name", font=FONT_BODY, fill="#6b6557")
        draw.text((S(110), y + S(72)), "Reason for leaving", font=FONT_BODY, fill="#6b6557")
        draw.line((S(110), y + S(26), S(250), y + S(26)), fill="#b0a58f", width=1)
        draw.line((S(280), y + S(26), S(470), y + S(26)), fill="#b0a58f", width=1)
        draw.line((S(500), y + S(26), S(586), y + S(26)), fill="#b0a58f", width=1)
        draw.line((S(110), y + S(66), S(330), y + S(66)), fill="#b0a58f", width=1)
        draw.line((S(360), y + S(66), S(586), y + S(66)), fill="#b0a58f", width=1)
        draw.line((S(110), y + S(96), S(586), y + S(96)), fill="#b0a58f", width=1)
        if not blank:
            draw.text((S(110), y + S(12)), row["employer"], font=FONT_BODY, fill="#2f2b23")
            draw.text((S(280), y + S(12)), row["address"], font=FONT_BODY, fill="#2f2b23")
            draw.text((S(500), y + S(12)), row["dates"], font=FONT_BODY, fill="#2f2b23")
            draw.text((S(110), y + S(52)), row["title"], font=FONT_BODY, fill="#2f2b23")
            draw.text((S(360), y + S(52)), row["supervisor"], font=FONT_BODY, fill="#2f2b23")
            draw.text((S(110), y + S(84)), row["reason"], font=FONT_BODY, fill="#2f2b23")

    draw.rectangle((S(104), S(760), S(596), S(790)), fill="#e5ddca", outline="#9d9278", width=1)
    draw.text((S(116), S(767)), "Section 3. Availability and Certification", font=FONT_SECTION, fill="#413828")

    draw.text((S(110), S(800)), "Earliest start date", font=FONT_BODY, fill="#6b6557")
    draw.text((S(335), S(800)), "Available weekends?", font=FONT_BODY, fill="#6b6557")
    draw.text((S(486), S(800)), "Badge renewal date", font=FONT_BODY, fill="#6b6557")
    draw.line((S(110), S(822), S(260), S(822)), fill="#b0a58f", width=1)
    draw.rectangle((S(335), S(818), S(349), S(832)), outline="#b0a58f", width=1)
    draw.rectangle((S(412), S(818), S(426), S(832)), outline="#b0a58f", width=1)
    draw.text((S(358), S(819)), "Yes", font=FONT_TINY, fill="#6b6557")
    draw.text((S(435), S(819)), "No", font=FONT_TINY, fill="#6b6557")
    draw.line((S(486), S(822), S(586), S(822)), fill="#b0a58f", width=1)

    if not blank:
        draw.text((S(110), S(808)), "04/27/2026", font=FONT_BODY, fill="#2f2b23")
        draw.text((S(494), S(808)), "05/15/2026", font=FONT_BODY, fill="#2f2b23")

    draw.text((S(110), S(844)), "Disciplinary action within last 24 months?", font=FONT_BODY, fill="#6b6557")
    draw.rectangle((S(420), S(840), S(434), S(854)), outline="#b0a58f", width=1)
    draw.rectangle((S(486), S(840), S(500), S(854)), outline="#b0a58f", width=1)
    draw.text((S(444), S(841)), "Yes", font=FONT_TINY, fill="#6b6557")
    draw.text((S(510), S(841)), "No", font=FONT_TINY, fill="#6b6557")
    if not blank:
        draw.text((S(110), S(860)), "Explain if yes on attached page.", font=FONT_TINY, fill="#6b6557")

    dense_y = S(878)
    draw.text((S(110), dense_y), "After a written offer, payroll records and I-9 support must be submitted within", font=FONT_MICRO, fill="#6b6557")
    prefix = "the shorter of 30 calendar days, 7 orientation days, or "
    draw.text((S(110), dense_y + S(16)), prefix, font=FONT_MICRO, fill="#6b6557")
    prefix_w = draw.textlength(prefix, font=FONT_MICRO)
    blank_x = S(110) + int(prefix_w)
    blank_w = S(24)
    blank_y = dense_y + S(16) + S(12)
    draw.line((blank_x, blank_y, blank_x + blank_w, blank_y), fill="#756a52", width=1)
    suffix_x = blank_x + blank_w + S(4)
    draw.text((suffix_x, dense_y + S(16)), "business days", font=FONT_MICRO, fill="#6b6557")
    draw.text((S(110), dense_y + S(30)), "after supervisor notice.", font=FONT_MICRO, fill="#6b6557")
    draw.text((S(110), dense_y + S(48)), "This deadline is separate from the 90-day review window and the 14-day badge reset.", font=FONT_MICRO, fill="#6b6557")
    if not blank:
        draw.text((blank_x + S(2), dense_y + S(12)), FIELDS[3].value, font=FONT_VALUE_SMALL, fill="#2f2b23")

    return image

In [ ]:
# Render blank template and filled form
blank = render_template(blank=True)
filled = render_template(blank=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 10))
axes[0].imshow(blank)
axes[0].set_title("Blank template")
axes[0].axis("off")

# Show filled form with field bounding boxes
axes[1].imshow(filled)
colors = ["#e74c3c", "#2ecc71", "#3498db", "#f39c12"]
for field, color in zip(FIELDS, colors):
    x, y, w, h = field.box
    rect = mpatches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor="none", linestyle="--")
    axes[1].add_patch(rect)
    axes[1].text(x, y - 8, field.label, fontsize=8, color=color, fontweight="bold")
axes[1].set_title("Filled form (fields highlighted)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("When the template is known, extraction = crop at fixed coordinates. No model needed.")

## 2. Corrupting the Form

Real documents arrive with printer artifacts, geometric distortion, and capture noise. We simulate each category to see how they affect extraction.

In [ ]:
# --- Conversion helpers ---

def to_cv(image: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

def from_cv(image: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

def to_float_rgb(image: Image.Image) -> np.ndarray:
    return np.asarray(image).astype(np.float32) / 255.0

def from_float_rgb(array: np.ndarray) -> Image.Image:
    return Image.fromarray(np.clip(array * 255.0, 0, 255).astype(np.uint8), mode="RGB")

def smooth_noise(width, height, rng, scale=16):
    small_w, small_h = max(2, width // scale), max(2, height // scale)
    base = rng.random((small_h, small_w), dtype=np.float32)
    return cv2.resize(base, (width, height), interpolation=cv2.INTER_CUBIC)

# --- Print artifacts ---

def apply_ink_fade(image, fade_strength, seed):
    if fade_strength <= 0: return image
    rng = np.random.default_rng(seed)
    rgb = np.asarray(image).astype(np.float32)
    brightness = rgb.mean(axis=2)
    ink_mask = np.clip((220 - brightness) / 220, 0, 1)
    fade_map = 0.55 + 0.45 * smooth_noise(image.width, image.height, rng, scale=26)
    adjust = 1.0 - fade_strength * ink_mask * (1.0 - fade_map)
    faded = rgb * adjust[..., None] + 255 * (1 - adjust[..., None])
    return Image.fromarray(np.clip(faded, 0, 255).astype(np.uint8), mode="RGB")

def apply_morphological_noise(image, amount, seed):
    if amount <= 0: return image
    img, rng = to_cv(image), np.random.default_rng(seed)
    k = 2 if amount < 0.5 else 3
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    dilated = cv2.erode(img, kernel, iterations=1)
    alpha = min(amount * 0.6, 0.4)
    blended = cv2.addWeighted(img, 1.0 - alpha, dilated, alpha, 0)
    for _ in range(int(amount * 800)):
        x, y = rng.integers(0, blended.shape[1]), rng.integers(0, blended.shape[0])
        shade = rng.integers(160, 230)
        blended[y, x] = [shade, shade, shade]
    return from_cv(blended)

def apply_print_streaks(image, strength, seed):
    if strength <= 0: return image
    rng = random.Random(seed)
    result = image.copy()
    draw = ImageDraw.Draw(result, "RGBA")
    w, h = image.size
    for _ in range(rng.randint(2, 5)):
        x = rng.randint(int(w * 0.05), int(w * 0.95))
        thickness = rng.randint(1, 3)
        alpha = int(strength * rng.randint(25, 60))
        y0, y1 = rng.randint(0, int(h * 0.1)), rng.randint(int(h * 0.85), h)
        draw.rectangle((x, y0, x + thickness, y1), fill=(40, 35, 30, alpha))
    return result.convert("RGB")

def apply_toner_band(image, strength, seed):
    if strength <= 0: return image
    rng = np.random.default_rng(seed)
    rgb = to_float_rgb(image)
    band = smooth_noise(image.width, image.height, rng, scale=80)
    rgb *= 1.0 - strength * 0.18 * band[..., None]
    return from_float_rgb(rgb)

def apply_print_pipeline(image, print_artifacts, seed):
    result = apply_ink_fade(image, print_artifacts.get("ink_fade", 0.0), seed)
    result = apply_morphological_noise(result, print_artifacts.get("morph_noise", 0.0), seed + 5)
    result = apply_print_streaks(result, print_artifacts.get("print_streaks", 0.0), seed + 7)
    result = apply_toner_band(result, print_artifacts.get("toner_band", 0.0), seed + 9)
    return result

# --- Paper artifacts ---

_texture_cache: dict = {}

def generate_procedural_texture(kind, width, height, seed):
    rng = np.random.default_rng(seed)
    base = np.ones((height, width, 3), dtype=np.float32)
    if kind == "paper":
        base[:] = [0.97, 0.95, 0.90]
        base *= 0.96 + 0.08 * smooth_noise(width, height, rng, scale=12)[..., None]
        base *= 0.98 + 0.04 * smooth_noise(width, height, rng, scale=40)[..., None]
    elif kind == "stain":
        overlay = np.ones_like(base)
        stain_mask = np.clip((smooth_noise(width, height, rng, scale=18) - 0.62) * 2.2, 0, 1)
        overlay[:] = [0.74, 0.67, 0.52]
        base = base * (1 - stain_mask[..., None] * 0.35) + overlay * stain_mask[..., None] * 0.35
    return from_float_rgb(base)

def load_texture_bank(kind, width, height):
    key = (kind, width, height)
    if key not in _texture_cache:
        _texture_cache[key] = [generate_procedural_texture(kind, width, height, seed=101 + i * 17) for i in range(3)]
    return _texture_cache[key]

def blend_multiply(base, overlay, strength):
    a, b = to_float_rgb(base), to_float_rgb(overlay)
    return from_float_rgb(a * ((1.0 - strength) + strength * b))

def blend_overlay(base, overlay, strength):
    a, b = to_float_rgb(base), to_float_rgb(overlay)
    return from_float_rgb(a * (1.0 - strength) + b * strength)

def apply_paper_texture(image, seed, stain_strength, paper_strength):
    paper = load_texture_bank("paper", *TEMPLATE_SIZE)[seed % 3]
    result = blend_multiply(image, paper, paper_strength)
    if stain_strength > 0:
        stain = load_texture_bank("stain", *TEMPLATE_SIZE)[(seed + 1) % 3]
        result = blend_overlay(result, stain, stain_strength)
    return result

def add_fold_shadows(image, strength, seed):
    if strength <= 0: return image
    rng = random.Random(seed)
    shaded = image.copy()
    draw = ImageDraw.Draw(shaded, "RGBA")
    w, h = image.size
    for _ in range(2):
        x = rng.randint(int(w * 0.1), int(w * 0.8))
        width = rng.randint(18, 34)
        alpha = int(45 * strength)
        draw.rectangle((x, int(h * 0.05), x + width, int(h * 0.95)), fill=(90, 80, 60, alpha))
        draw.rectangle((x + width, int(h * 0.05), x + width + 6, int(h * 0.95)), fill=(255, 255, 255, int(26 * strength)))
    return shaded.convert("RGB")

def apply_paper_pipeline(image, paper_artifacts, seed):
    result = apply_paper_texture(image, seed=seed,
        stain_strength=paper_artifacts.get("stain_strength", 0.0),
        paper_strength=paper_artifacts.get("paper_texture", 0.0))
    result = add_fold_shadows(result, paper_artifacts.get("fold_shadow", 0.0), seed)
    return result

# --- Geometry ---

def warp_perspective(image, offsets):
    x0, y0 = PAGE_BOX["x"], PAGE_BOX["y"]
    x1, y1 = x0 + PAGE_BOX["w"], y0 + PAGE_BOX["h"]
    src = np.float32([[x0, y0], [x1, y0], [x1, y1], [x0, y1]])
    dst = np.float32([[x0 + S(o[0]), y0 + S(o[1])] for o in offsets])
    matrix = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(to_cv(image), matrix, TEMPLATE_SIZE, flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(238, 227, 202))
    return from_cv(warped)

def wave_displacement(image, amplitude, wavelength, seed):
    if amplitude <= 0: return image
    src = to_cv(image)
    h, w = src.shape[:2]
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    phase = seed * 0.37
    map_x = xs + (amplitude * np.sin(2 * np.pi * ys / wavelength + phase)).astype(np.float32)
    map_y = ys + (amplitude * 0.55 * np.sin(2 * np.pi * xs / (wavelength * 1.4) + phase * 0.7)).astype(np.float32)
    warped = cv2.remap(src, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(238, 227, 202))
    return from_cv(warped)

def rotate_scale(image, angle, scale_x, scale_y):
    w, h = TEMPLATE_SIZE
    src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
    center = np.array([w / 2.0, h / 2.0], dtype=np.float32)
    rad = math.radians(angle)
    rot = np.array([[math.cos(rad), -math.sin(rad)], [math.sin(rad), math.cos(rad)]], dtype=np.float32)
    sc = np.array([[scale_x, 0], [0, scale_y]], dtype=np.float32)
    transform = rot @ sc
    dst = np.float32([transform @ (p - center) + center for p in src])
    matrix = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(to_cv(image), matrix, TEMPLATE_SIZE, flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(238, 227, 202))
    return from_cv(warped)

def shrink_and_offset(image, scale, offset_x, offset_y, background=(236, 229, 210)):
    if abs(scale - 1.0) < 1e-4 and offset_x == 0 and offset_y == 0: return image
    src = to_cv(image)
    h, w = src.shape[:2]
    matrix = np.float32([[scale, 0.0, offset_x + (1.0 - scale) * w / 2.0],
                         [0.0, scale, offset_y + (1.0 - scale) * h / 2.0]])
    warped = cv2.warpAffine(src, matrix, TEMPLATE_SIZE, flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=background)
    return from_cv(warped)

def apply_geometry_pipeline(image, geometry, seed):
    result = image
    if geometry.get("perspective"):
        result = warp_perspective(result, geometry["perspective"])
    angle, sx, sy = geometry.get("angle", 0.0), geometry.get("scale_x", 1.0), geometry.get("scale_y", 1.0)
    if angle or sx != 1.0 or sy != 1.0:
        result = rotate_scale(result, angle=angle, scale_x=sx, scale_y=sy)
    if geometry.get("wave_amplitude", 0.0):
        result = wave_displacement(result, amplitude=geometry["wave_amplitude"], wavelength=geometry.get("wave_length", 240.0), seed=seed)
    fs = geometry.get("frame_scale", 1.0)
    ox, oy = geometry.get("offset_x", 0), geometry.get("offset_y", 0)
    if fs != 1.0 or ox or oy:
        result = shrink_and_offset(result, scale=fs, offset_x=ox, offset_y=oy)
    return result

# --- Capture ---

def apply_lighting_gradient(image, strength):
    if strength <= 0: return image
    rgb = to_float_rgb(image)
    h, w = rgb.shape[:2]
    xv, yv = np.meshgrid(np.linspace(0, 1, w, dtype=np.float32), np.linspace(0, 1, h, dtype=np.float32))
    rgb *= (1.0 - strength * (0.55 * xv + 0.45 * yv))[..., None]
    return from_float_rgb(rgb)

def apply_camera_shadow(image, strength):
    if strength <= 0: return image
    rgb = to_float_rgb(image)
    h, w = rgb.shape[:2]
    xv, yv = np.meshgrid(np.linspace(-1, 1, w, dtype=np.float32), np.linspace(-1, 1, h, dtype=np.float32))
    vignette = np.exp(-(xv**2 + (yv * 1.2)**2) / 1.1)
    rgb *= (1.0 - strength * (1.0 - vignette))[..., None]
    return from_float_rgb(rgb)

def apply_motion_blur(image, radius, angle_deg):
    if radius <= 0: return image
    size = max(3, int(round(radius * 6))) | 1
    kernel = np.zeros((size, size), dtype=np.float32)
    kernel[size // 2, :] = 1.0
    center = (size / 2 - 0.5, size / 2 - 0.5)
    kernel = cv2.warpAffine(kernel, cv2.getRotationMatrix2D(center, angle_deg, 1.0), (size, size))
    s = kernel.sum()
    if s > 0: kernel /= s
    return from_cv(cv2.filter2D(to_cv(image), -1, kernel))

def apply_focus_falloff(image, radius, seed):
    if radius <= 0: return image
    rng = random.Random(seed)
    sharp = to_float_rgb(image)
    heavy = to_float_rgb(image.filter(ImageFilter.GaussianBlur(radius=radius)))
    h, w = sharp.shape[:2]
    xv, yv = np.meshgrid(np.linspace(0, 1, w, dtype=np.float32), np.linspace(0, 1, h, dtype=np.float32))
    cx, cy = 0.46 + rng.uniform(-0.06, 0.08), 0.42 + rng.uniform(-0.05, 0.07)
    sx, sy = 0.22 + rng.uniform(-0.03, 0.04), 0.26 + rng.uniform(-0.04, 0.05)
    mask = np.clip(0.25 + 0.85 * np.exp(-(((xv - cx)**2) / (2 * sx**2) + ((yv - cy)**2) / (2 * sy**2))), 0, 1)[..., None]
    return from_float_rgb(sharp * mask + heavy * (1 - mask))

def jpeg_roundtrip(image, quality):
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def apply_capture_pipeline(image, capture, seed):
    result = apply_lighting_gradient(image, capture.get("lighting_gradient", 0.0))
    result = apply_camera_shadow(result, capture.get("camera_shadow", 0.0))
    blur = capture.get("blur", 0.0)
    if blur:
        result = apply_motion_blur(result, radius=max(0.6, blur * 0.9), angle_deg=8 + (seed % 9) * 7)
        result = apply_focus_falloff(result, radius=blur * 1.35, seed=seed + 19)
    q = capture.get("jpeg_quality")
    if q: result = jpeg_roundtrip(result, quality=q)
    return result

# --- Orchestrator ---

def apply_profile(profile, image, seed):
    result = image.copy()
    result = apply_print_pipeline(result, profile.print_artifacts, seed)
    result = apply_paper_pipeline(result, profile.paper_artifacts, seed)
    result = apply_geometry_pipeline(result, profile.geometry, seed)
    result = apply_capture_pipeline(result, profile.capture, seed)
    return result

print("Corruption functions loaded.")

In [ ]:
NO_PAPER = {"paper_texture": 0.0, "stain_strength": 0.0, "fold_shadow": 0.0, "random_lines": 0, "random_dots": 0}
NO_GEO = {"angle": 0.0, "scale_x": 1.0, "scale_y": 1.0, "perspective": None, "wave_amplitude": 0.0, "wave_length": 240.0}
NO_CAP = {"blur": 0.0, "jpeg_quality": None, "lighting_gradient": 0.0, "camera_shadow": 0.0}

PROFILES = [
    CorruptionProfile("clean", "No corruption.",
        {"ink_fade": 0.0, "morph_noise": 0.0, "print_streaks": 0.0, "toner_band": 0.0}, NO_PAPER, NO_GEO, NO_CAP),
    CorruptionProfile("print", "Printer artifacts: ink bleed, roller streaks, toner banding.",
        {"ink_fade": 0.10, "morph_noise": 0.12, "print_streaks": 0.10, "toner_band": 0.06}, NO_PAPER, NO_GEO, NO_CAP),
    CorruptionProfile("rotation", "Mild skewed scan.",
        {"ink_fade": 0.03, "morph_noise": 0.0, "print_streaks": 0.0, "toner_band": 0.02},
        {"paper_texture": 0.02, "stain_strength": 0.0, "fold_shadow": 0.0, "random_lines": 0, "random_dots": 0},
        {"angle": 3.2, "scale_x": 1.0, "scale_y": 1.0, "perspective": None, "wave_amplitude": 0.0, "wave_length": 240.0}, NO_CAP),
    CorruptionProfile("phone_photo", "Phone photo with tilt, background, and blur.",
        {"ink_fade": 0.05, "morph_noise": 0.0, "print_streaks": 0.03, "toner_band": 0.03},
        {"paper_texture": 0.05, "stain_strength": 0.0, "fold_shadow": 0.0, "random_lines": 0, "random_dots": 0},
        {"angle": 1.6, "scale_x": 1.0, "scale_y": 1.0, "perspective": [(-10, 12), (16, -8), (20, 18), (-12, -6)],
         "wave_amplitude": 0.0, "wave_length": 240.0, "frame_scale": 0.88, "offset_x": 18, "offset_y": 22},
        {"blur": 0.55, "jpeg_quality": None, "lighting_gradient": 0.06, "camera_shadow": 0.08}),
    CorruptionProfile("combo", "Everything: low ink, stains, warp, blur, JPEG.",
        {"ink_fade": 0.16, "morph_noise": 0.15, "print_streaks": 0.12, "toner_band": 0.08},
        {"paper_texture": 0.10, "stain_strength": 0.10, "fold_shadow": 0.30, "random_lines": 0, "random_dots": 0},
        {"angle": 2.4, "scale_x": 1.03, "scale_y": 0.97, "perspective": [(-12, 16), (18, -8), (22, 20), (-18, -12)],
         "wave_amplitude": 2.8, "wave_length": 210.0},
        {"blur": 0.8, "jpeg_quality": 35, "lighting_gradient": 0.14, "camera_shadow": 0.12}),
]

# Apply each profile and display
random.seed(7)
np.random.seed(7)

variant_images = {}
fig, axes = plt.subplots(1, len(PROFILES), figsize=(4 * len(PROFILES), 6))
for i, profile in enumerate(PROFILES):
    variant = apply_profile(profile, filled, seed=400 + i * 13)
    variant_images[profile.id] = variant
    axes[i].imshow(variant)
    axes[i].set_title(profile.id, fontsize=11)
    axes[i].axis("off")
plt.suptitle("Corruption profiles applied to the filled form", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Why Fixed Coordinates Break

Without registration, we crop field regions using the template's pixel coordinates. When the scan is rotated or warped, the crop lands in the wrong place.

In [ ]:
def crop_with_padding(image: np.ndarray, field: Field) -> np.ndarray:
    x, y, w, h = field.box
    pad_map = {"name": (6, 8, 6, 8), "position": (6, 8, 6, 8),
               "address": (0, 12, 12, 12), "records_days": (2, 6, 8, 2)}
    left, top, right, bottom = pad_map.get(field.id, (8, 8, 8, 8))
    x0, y0 = max(0, x - left), max(0, y - top)
    x1, y1 = min(image.shape[1], x + w + right), min(image.shape[0], y + h + bottom)
    return image[y0:y1, x0:x1]


# Crop "name" field from clean, rotation, and combo WITHOUT registration
demo_variants = ["clean", "rotation", "combo"]
field = FIELDS[0]  # name

fig, axes = plt.subplots(1, len(demo_variants), figsize=(15, 2))
for i, vid in enumerate(demo_variants):
    crop = crop_with_padding(to_cv(variant_images[vid]), field)
    axes[i].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{vid} — fixed crop of '{field.label}'")
    axes[i].axis("off")
plt.suptitle("Cropping with fixed template coordinates (no registration)", fontsize=12)
plt.tight_layout()
plt.show()
print("The rotation and combo crops miss the actual field content entirely.")

## 4. Image Registration (ORB + RANSAC)

Image registration aligns a distorted scan back to the template. We detect ORB keypoints in both images, match them with a brute-force Hamming matcher, and compute a homography using RANSAC to warp the scan onto the template grid.

In [ ]:
def match_and_register(template, target):
    """Returns (registered_image_or_None, match_visualization, stats_dict)."""
    tg = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)
    sg = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=5000, fastThreshold=5)
    kp1, des1 = orb.detectAndCompute(tg, None)
    kp2, des2 = orb.detectAndCompute(sg, None)
    if des1 is None or des2 is None:
        return None, np.hstack([template, target]), {"ok": False, "reason": "no_descriptors"}

    good = []
    for pair in cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des1, des2, k=2):
        if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance:
            good.append(pair[0])
    if len(good) < 12:
        vis = cv2.drawMatches(template, kp1, target, kp2, good, None,
            matchColor=(64, 196, 120), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        return None, vis, {"ok": False, "matches": len(good)}

    src = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
    if H is None:
        return None, np.hstack([template, target]), {"ok": False, "matches": len(good)}

    # Visualization: sample up to 28 inlier matches
    inliers = [m for m, keep in zip(good, mask.ravel()) if keep]
    chosen = sorted(inliers, key=lambda m: m.distance)
    if len(chosen) > 28: chosen = chosen[::max(1, len(chosen) // 28)][:28]
    vis = cv2.drawMatches(template, kp1, target, kp2, chosen, None,
        matchColor=(64, 196, 120), singlePointColor=(215, 186, 120),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    registered = cv2.warpPerspective(target, H, TEMPLATE_SIZE, flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(238, 227, 202))
    inlier_count = int(mask.sum())
    return registered, vis, {"ok": True, "matches": len(good), "inliers": inlier_count,
                             "inlier_ratio": round(inlier_count / max(len(good), 1), 3)}


# Run registration on all variants
template_cv = to_cv(blank)
registrations = {}

for profile in PROFILES:
    img = variant_images[profile.id]
    registered, vis, stats = match_and_register(template_cv, to_cv(img))
    registrations[profile.id] = {"registered": registered, "vis": vis, "stats": stats}
    status = f"matches={stats.get('matches', 0)}, inliers={stats.get('inliers', 0)}, ratio={stats.get('inlier_ratio', 0)}"
    print(f"  {profile.id:>12}: {status}")

# Show match visualizations
fig, axes = plt.subplots(len(PROFILES), 1, figsize=(16, 4 * len(PROFILES)))
for i, profile in enumerate(PROFILES):
    vis = registrations[profile.id]["vis"]
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{profile.id} — ORB matches (green = inliers used for homography)")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 5. Crop Fields and Run OCR

With registered images, we crop each field at its template coordinates and run OCR. We use **doctr** as the primary engine with **tesseract** as fallback for isolated short text.

In [ ]:
import tempfile
from doctr.models import ocr_predictor
from doctr.io import DocumentFile
import pytesseract

predictor = ocr_predictor(pretrained=True)

def ocr_crop(crop_bgr: np.ndarray) -> str:
    """Run doctr on a crop, fall back to tesseract if empty."""
    bordered = cv2.copyMakeBorder(crop_bgr, 10, 10, 10, 10,
        borderType=cv2.BORDER_CONSTANT, value=(255, 255, 255))
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        cv2.imwrite(f.name, bordered)
        doc = DocumentFile.from_images(f.name)
    result = predictor(doc)
    text = " ".join(result.render().split())
    if not text.strip():
        text = " ".join(pytesseract.image_to_string(bordered, config="--psm 7").split())
    return text

def normalize_text(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", value.lower())

def texts_match(expected: str, ocr_text: str) -> bool:
    return normalize_text(expected) == normalize_text(ocr_text)

print("OCR engine ready.")

In [ ]:
# Run OCR on all registered variants and build results table
results = []

for profile in PROFILES:
    reg = registrations[profile.id]["registered"]
    if reg is None:
        for field in FIELDS:
            results.append({"variant": profile.id, "field": field.id, "expected": field.value, "ocr": "", "correct": False})
        continue
    for field in FIELDS:
        crop = crop_with_padding(reg, field)
        ocr_text = ocr_crop(crop)
        correct = texts_match(field.value, ocr_text)
        results.append({"variant": profile.id, "field": field.id, "expected": field.value, "ocr": ocr_text, "correct": correct})

# Print summary table
print(f"{'Variant':>12}  {'Field':>15}  {'Correct':>7}  {'Expected':<40}  {'OCR output'}")
print("-" * 120)
for r in results:
    mark = "  ✓" if r["correct"] else "  ✗"
    print(f"{r['variant']:>12}  {r['field']:>15}  {mark:>7}  {r['expected']:<40}  {r['ocr']}")

# Score summary
print("\n--- Score summary ---")
for profile in PROFILES:
    pr = [r for r in results if r["variant"] == profile.id]
    ok = sum(1 for r in pr if r["correct"])
    print(f"  {profile.id:>12}: {ok}/{len(pr)}")

In [ ]:
# Visualize crops for clean vs combo
for vid in ["clean", "combo"]:
    reg = registrations[vid]["registered"]
    if reg is None:
        print(f"{vid}: registration failed, no crops to show")
        continue
    fig, axes = plt.subplots(1, len(FIELDS), figsize=(16, 2))
    fig.suptitle(f"Field crops — {vid}", fontsize=12)
    for i, field in enumerate(FIELDS):
        crop = crop_with_padding(reg, field)
        axes[i].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        r = next(r for r in results if r["variant"] == vid and r["field"] == field.id)
        color = "green" if r["correct"] else "red"
        axes[i].set_title(f"{field.id}\nocr: {r['ocr'][:30]}", fontsize=8, color=color)
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

## 6. What We Learned

**What works:** On clean scans, the template pipeline is perfect — fixed coordinates + OCR gives 4/4 fields correct. No model training needed.

**What breaks:** Even mild geometric distortion (rotation, perspective) shifts crop regions. Heavy corruption (blur, JPEG, combined noise) degrades OCR even after successful registration. The tiny `records_days` field is hardest — single characters in small regions are at the edge of what OCR can detect.

**The fundamental limitation:** This pipeline assumes every document matches the template exactly. Any layout change — a revised form, a different printer, a shifted section — silently extracts wrong fields. Registration fixes geometry but can't fix structural changes.

**Next step:** Room 2 replaces fixed coordinates with a learned field detector (ConvNet), so the pipeline can find fields regardless of layout.

> [Back to the blog post](https://zzsi.github.io/blog/posts/20260331-practical-doc-ai/) — [Room 1 interactive exhibit](https://zzsi.github.io/blog/posts/20260331-practical-doc-ai/museum/room1-templates.html)